In [0]:

CREATE SCHEMA IF NOT EXISTS bronze_education;


In [0]:
DROP TABLE IF EXISTS bronze_education.education_sun;

-- Create Delta table with Column Mapping enabled
CREATE TABLE if not exists university_sun
OPTIONS (delta.columnMapping.mode = 'name', delta.minReaderVersion = '2', delta.minWriterVersion = '5')

-- Ingest data from ADLS
AS SELECT * FROM read_files("abfss://landing@adlmlabourmarket.dfs.core.windows.net/raw/uni_education/uni_edu_sun.csv",format => "csv",header => true,sep => ";",inferSchema => true);

-- Verify data
SELECT * FROM university_sun LIMIT 5;

In [0]:
SELECT DISTINCT `lärosäte` FROM bronze_education.university_sun

In [0]:
%python
pip install pyjstat

##Below Dataset

>Studerande och examinerade inom yrkeshögskolan. Antal examinerade efter kön, utbildningens inriktning, region där utbildningen bedrivs, utbildningens studietakt, utbildningens studieform, utbildningens längd, utbildningens examenstyp och år

In [0]:
%python
from pyjstat import pyjstat
import requests

# YH Graduates by Region
url = "https://statistikdatabasen.scb.se/api/v2/tables/TAB5770/data?lang=sv&valueCodes[ContentsCode]=000004Z6&valueCodes[Kon]=2,1&valueCodes[UtbildnInriktn]=1M,2M,3M,4M,5M,6M,7M,8M,9M,11M,12M,13M,14M,15M&valueCodes[RegionUtb]=01,03,04,05,06,07,08,09,10,12,13,14,17,18,19,20,21,22,23,24,25,99&valueCodes[UtbTakt]=TOTHD&valueCodes[UtbFormYH]=BU,DI&valueCodes[UtbLangd]=TOT&valueCodes[UtbExam]=TOTTYP&valueCodes[Tid]=2014,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024&codelist[UtbildnInriktn]=vs_AUtbOmrMYH&codelist[RegionUtb]=vs_ALANYH"

response = requests.get(url)
dataset = pyjstat.Dataset.read(response.text)
df = dataset.write('dataframe')

df.columns = [col.replace(' ', '_').replace('(', '_').replace(')', '_').replace(';', '_').replace(',', '_').replace('{', '_').replace('}', '_').replace('=', '_').replace('\n', '_').replace('\t', '_') for col in df.columns]; spark_df = spark.createDataFrame(df)
spark.sql("DROP TABLE IF EXISTS bronze_education.yh_region")
spark_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("bronze_education.yh_region_subject")

print("✅ Done:", spark_df.count(), "rows")
spark_df.show(5)

In [0]:
%python
from pyjstat import pyjstat
import requests

# YH Graduates by Region
url = "https://statistikdatabasen.scb.se/api/v2/tables/TAB5774/data?lang=sv&valueCodes[ContentsCode]=000004ZM&valueCodes[Kon]=2,1&valueCodes[UtbildnInriktn]=1M,2M,3M,4M,5M,6M,7M,8M,9M,11M,12M,13M,14M,15M,16&valueCodes[Alder]=totalt&valueCodes[Tid]=2016,2017,2018,2019,2020,2021,2022,2023,2024&codelist[UtbildnInriktn]=vs_AUtbOmrMYH"

response = requests.get(url)
dataset = pyjstat.Dataset.read(response.text)
df = dataset.write('dataframe')

df.columns = [col.replace(' ', '_').replace('(', '_').replace(')', '_').replace(';', '_').replace(',', '_').replace('{', '_').replace('}', '_').replace('=', '_').replace('\n', '_').replace('\t', '_') for col in df.columns]; spark_df = spark.createDataFrame(df)


spark_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("bronze_education.yh_subject")

print("✅ Done:", spark_df.count(), "rows")
spark_df.show(5)

In [0]:

CREATE OR REPLACE TABLE bronze_education.university_lan_lookup (`Lärosäte` STRING, `Län` STRING
);

-- Insert all universities and their läns
INSERT INTO bronze_education.university_lan_lookup VALUES
('Akademi för Ledarskap och Teologi', 'Östergötland'),
('Blekinge tekniska högskola', 'Blekinge'),
('Chalmers tekniska högskola', 'Västra Götaland'),
('Enskilda Högskolan Stockholm', 'Stockholm'),
('Försvarshögskolan', 'Stockholm'),
('Gymnastik- och idrottshögskolan', 'Stockholm'),
('Göteborgs universitet', 'Västra Götaland'),
('Handelshögskolan i Stockholm', 'Stockholm'),
('Högskolan Dalarna', 'Dalarna'),
('Högskolan i Borås', 'Västra Götaland'),
('Högskolan i Gävle', 'Gävleborg'),
('Högskolan i Halmstad', 'Halland'),
('Högskolan i Skövde', 'Västra Götaland'),
('Högskolan Kristianstad', 'Skåne'),
('Högskolan Väst', 'Västra Götaland'),
('Johannelunds teologiska högskola', 'Stockholm'),
('Karlstads universitet', 'Värmland'),
('Karolinska institutet', 'Stockholm'),
('Konstfack', 'Stockholm'),
('Kungl. Musikhögskolan i Stockholm', 'Stockholm'),
('Kungl. Tekniska högskolan', 'Stockholm'),
('Linköpings universitet', 'Östergötland'),
('Linnéuniversitetet', 'Kalmar'),
('Luleå tekniska universitet', 'Norrbotten'),
('Lunds universitet', 'Skåne'),
('Malmö universitet', 'Skåne'),
('Marie Cederschiöld högskola', 'Stockholm'),
('Mittuniversitetet', 'Västernorrland'),
('Mälardalens universitet', 'Södermanland'),
('Newmaninstitutet', 'Stockholm'),
('Röda Korsets högskola', 'Stockholm'),
('Sophiahemmet högskola', 'Stockholm'),
('Stiftelsen Högskolan i Jönköping', 'Jönköping'),
('Stockholms konstnärliga högskola', 'Stockholm'),
('Stockholms Musikpedagogiska Institut', 'Stockholm'),
('Stockholms universitet', 'Stockholm'),
('Sveriges lantbruksuniversitet', 'Uppsala'),
('Södertörns högskola', 'Stockholm'),
('Umeå universitet', 'Västernorrland'),
('Uppsala universitet', 'Uppsala'),
('Världssjöfartsuniversitetet', 'Stockholm'),
('Örebro universitet', 'Örebro'),
('Örebro teologiska högskola', 'Örebro'),
('Ersta Sköndal Bräcke högskola', 'Stockholm'),
('Mälardalens högskola', 'Södermanland'),
('Teologiska Högskolan, Stockholm', 'Stockholm'),
('Malmö högskola', 'Skåne');

-- Verify the data loaded
SELECT * FROM bronze_education.university_lan_lookup;